# 11.4 The Transformer Architecture

Let us first see it in pseudo code

```
def self_attention(input_sequence):
    output = np.zeros(shape=input_sequence.shape)
    for i, pivot_vector in enumerate(input_sequence): # Iterate over each token in the input
        scores = np.zeros(shape=(len(input_sequence), ))
        for j, vector in enumerate(input_sequence):
            scores[j] = np.dot(pivot_vector, vector.T)
        scores /= np.sqrt(input_sequence.shape[1]);
        scores = softmax(scores)
        new_pivot_representation = np.zeros(shape=pivot_vector.shape)
        for j, vector in enumerate(input_sequence):
            new_pivot_representation += vector * scores[j]
        output[i] = new_pivot_representation
    return output
```

But in keras, we don't have to use it like this. It has a built-in layer for us `MultiHeadAttention`

```
num_heads = 4
embed_dim = 256
mha_layer = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
outputs = mha_layer(inputs, inputs, inputs)
```

In [1]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  5748k      0  0:00:14  0:00:14 --:--:-- 11.1M


In [2]:
!rm -r aclImdb/train/unsup/

In [3]:
!cat aclImdb/train/pos/1103_10.txt
print()
!cat aclImdb/train/neg/11050_1.txt

This film has some of the greatest comedic dialog and memorable quotes ever assembled in one film! The plot is somewhat lacking, but the delightful quips are enough to make up the difference. This is a timeless movie for all ages that is sure to please. As a cinematic art form it is highly entertaining; and with major stars like Cary Grant, Myrna Loy, and Melvyn Douglas... how could you go wrong? <br /><br />Comedic dialog and timeing such as this has long been undervalued, and is very difficult to imitate. A good example of this is seen in the 1986 knockoff of this film: The Money Pit, with Tom Hanks and Shelley Long. Despite the talent and physical comedy of these stars, the film dragged and received poor reviews and viewer comments. Achieving true comedic dialog is an art.
Terrible use of scene cuts. All continuity is lost, either by awful scripting or lethargic direction. That villainous robot... musta been a jazz dancer? Also, one of the worst sound tracks I've ever heard (monolog

In [4]:
# Essential Libraries to import

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.layers import TextVectorization

In [5]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    # print(type(files))
    random.Random(1337).shuffle(files)
    num_val_samples = int (0.2 * len(files))
    val_files = files[-num_val_samples: ]

    for fname in val_files:
        shutil.move(train_dir / category / fname,
                    val_dir / category / fname)

In [6]:
from tensorflow import keras

batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size=batch_size
)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val",
    batch_size=batch_size
)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test",
    batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [8]:
text_only_train_ds = train_ds.map(lambda x, y: x)

In [11]:
from tensorflow.keras import layers
from keras.layers import TextVectorization

max_length = 600
max_tokens = 20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=max_length
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

In [16]:
class TransformerEncoder(keras.layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim # Size of input token vectors
        self.dense_dim = dense_dim # Size of inner dense
        self.num_heads = num_heads # Number of attention heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [layers.Dense(units=dense_dim, activation='relu'),
             layers.Dense(units=embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):
        # The mask that will be generated by the embedding layer will be 2D
        # But the attention layer expect the mask to be 3D or 3D.
        # So we expand its rank
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        # Implementing serialization so that we can save the model
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "dense_dim": self.dense_dim,
            "num_heads": self.num_heads,
        })
        return config

In [17]:
vocab_size = 20000
embed_dim = 256
num_heads = 2
dense_dim = 32

inputs = keras.Input(shape=(None, ), dtype='int64')
x = layers.Embedding(vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='transformer_encoder.keras',
        save_best_only=True
    )
]

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_1           │ (None, None, 256)      │       543,776 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,664,033 (21.61 MB)

 Trainable params: 5,664,033 (21.61 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 67s 95ms/step - accuracy: 0.6173 - loss: 0.7596 - val_accuracy: 0.8368 - val_loss: 0.3672
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 94ms/step - accuracy: 0.8313 - loss: 0.3801 - val_accuracy: 0.8474 - val_loss: 0.3530
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 95ms/step - accuracy: 0.8582 - loss: 0.3292 - val_accuracy: 0.8662 - val_loss: 0.3111
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 61s 98ms/step - accuracy: 0.8717 - loss: 0.2992 - val_accuracy: 0.8734 - val_loss: 0.3033
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 101ms/step - accuracy: 0.8826 - loss: 0.2777 - val_accuracy: 0.8742 - val_loss: 0.3004
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 99ms/step - accuracy: 0.8967 - loss: 0.2511 - val_accuracy: 0.8758 - val_loss: 0.3009
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 62s 100ms/step - accuracy: 0.9057 - loss: 0.2298 - val_accuracy: 0.8780 - val_loss: 0.3054
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 103ms/step - accuracy: 0.9146 - loss: 0.2129

In [20]:
model = keras.models.load_model(
    'transformer_encoder.keras',
    custom_objects={"TransformerEncoder": TransformerEncoder}
)
print(f"Test Accuracy: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 17ms/step - accuracy: 0.8714 - loss: 0.3073
Test Accuracy: 0.873


We know that transformer architecture include positional information in it. But we haven't seen it yet. How does it do that. In the actual paper they used something else, but here we are using something simple and easy to grasp.

In [23]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim
        )
        self.positional_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim
        )
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.positional_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super().get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config


In [26]:
vocab_size = 20000
sequence_length = 600
embed_dim = 256
num_heads = 2
dense_dim = 32

inputs = keras.Input(shape=(None, ), dtype='int64')
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(inputs)
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(units=1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='full_transformer_encoder.keras',
        save_best_only=True
    )
]

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding_3          │ (None, None, 256)      │     5,273,600 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_3           │ (None, None, 256)      │       543,776 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_3          │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,817,633 (22.19 MB)

 Trainable params: 5,817,633 (22.19 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 70s 98ms/step - accuracy: 0.5993 - loss: 0.8089 - val_accuracy: 0.8022 - val_loss: 0.4254
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 100ms/step - accuracy: 0.8055 - loss: 0.4265 - val_accuracy: 0.8238 - val_loss: 0.3813
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 101ms/step - accuracy: 0.8341 - loss: 0.3671 - val_accuracy: 0.8358 - val_loss: 0.3609
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 63s 101ms/step - accuracy: 0.8558 - loss: 0.3248 - val_accuracy: 0.8458 - val_loss: 0.3502
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 64s 102ms/step - accuracy: 0.8763 - loss: 0.2904 - val_accuracy: 0.8306 - val_loss: 0.4023
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 106ms/step - accuracy: 0.8930 - loss: 0.2566 - val_accuracy: 0.8450 - val_loss: 0.3593
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 79s 101ms/step - accuracy: 0.9178 - loss: 0.2149 - val_accuracy: 0.8442 - val_loss: 0.3580
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 84s 104ms/step - accuracy: 0.9314 - loss: 0.

In [29]:
model = keras.models.load_model(
    'full_transformer_encoder.keras',
    custom_objects={'TransformerEncoder': TransformerEncoder,
                    'PositionalEmbedding': PositionalEmbedding}
    )
print(f"Test Accuracy: {model.evaluate(int_test_ds)[1]:.3f}")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'positional_embedding_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:393: UserWarning: `build()` was called on layer 'transformer_encoder_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 19ms/step - accuracy: 0.8492 - loss: 0.3426
Test Accuracy: 0.850


 It turns out that when approaching a new text-classification task, you should pay
 close attention to the ratio between the number of samples in your training data and
 the mean number of words per sample. If that ratio is small—less
 than 1,500—then the bag-of-bigrams model will perform better (and as a bonus, it will
 be much faster to train and to iterate on too). If that ratio is **higher than 1,500**, then
 you should go with a sequence model. In other words, sequence models work best
 when lots of training data is available and when each sample is relatively short.

# 11.5 Sequence to Sequence Learning: Translation

Skipped for now!!!